# Read Silver Tables

In [0]:
from pyspark.sql.functions import *

CATALOG = "workspace"
SCHEMA = "retail"

customers = spark.table(f"{CATALOG}.{SCHEMA}.silver_customers")
orders = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders")
products = spark.table(f"{CATALOG}.{SCHEMA}.silver_products")

# Create Dimension Tables

In [0]:
dim_customer = customers.select(
    "customer_id",
    "name",
    "email",
    "city",
    "state"
)

dim_customer.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_customer")

# Product Dimension

In [0]:
dim_product = products.select(
    "product_id",
    "product_name",
    "category",
    "price"
)

dim_product.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.dim_product")

# Create Fact Table

In [0]:
fact_sales = (
    orders.alias("o")
    .join(products.alias("p"), "product_id")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("o.product_id"),
        col("o.order_date"),
        col("o.quantity"),
        col("p.price"),
        (col("o.quantity") * col("p.price")).alias("sales_amount")
    )
)

fact_sales.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.fact_sales")

# Business KPI Table

In [0]:
gold_daily_sales = (
    fact_sales
    .groupBy("order_date")
    .agg(
        sum("sales_amount").alias("total_revenue"),
        sum("quantity").alias("units_sold"),
        countDistinct("customer_id").alias("unique_customers"),
        count("*").alias("total_orders")
    )
)

gold_daily_sales.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_daily_sales")

# Category Revenue

In [0]:
gold_category_sales = (
    fact_sales
    .join(dim_product, "product_id")
    .groupBy("category")
    .agg(
        round(sum("sales_amount"),2).alias("revenue")
    )
    .orderBy(desc("revenue"))
)

display(gold_category_sales)

# Top Customers

In [0]:
gold_customer_sales = (
    fact_sales
    .groupBy("customer_id")
    .agg(
        round(sum("sales_amount"),2).alias("lifetime_value")
    )
    .orderBy(desc("lifetime_value"))
    .limit(10)
)

display(gold_customer_sales)